In [10]:
import re
from pathlib import Path

with open(
    r"C:\Users\LINGRAM\Documents\PBI Test Files\5 Year Look Back (1).SemanticModel\definition\expressions.tmdl",
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

expressions = []

for line in text.splitlines():
    if line.startswith("expression "):
        expr_name = line.split("=", 1)[0]
        expr_name = expr_name[len("expression "):].strip().strip("'")
        expressions.append(expr_name)


def find_expression_references(expressions, tables_folder):
    references = {
        expr: {
            "file_count": 0,
            "files": set()
        }
        for expr in expressions
    }

    for tmdl_file in Path(tables_folder).rglob("*.tmdl"):
        with open(tmdl_file, "r", encoding="utf-8") as f:
            contents = f.read()

        for expr in expressions:
            if expr in contents:
                references[expr]["files"].add(tmdl_file.name)

    # Populate file_count after all files have been scanned
    for expr in references:
        references[expr]["file_count"] = len(references[expr]["files"])

    return references

Non_Used_Expressions = find_expression_references(expressions, r"C:\Users\LINGRAM\Documents\PBI Test Files\5 Year Look Back (1).SemanticModel\definition\tables")
for expr, info in Non_Used_Expressions.items():
    if info["file_count"] == 0:
        print(expr)
print(len(Non_Used_Expressions))

Parameter20
Transform Sample File (20)
Parameter21
Transform Sample File (21)
Sample File (21)
Transform File (21)
Parameter22
Transform Sample File (22)
Sample File (22)
Transform File (22)
Parameter23
Transform Sample File (23)
Sample File (23)
Transform File (23)
Parameter24
Transform Sample File (24)
Sample File (24)
Transform File (24)
Parameter25
Transform Sample File (25)
Sample File (25)
Transform File (25)
Errors in Final YTD
Parameter2
Transform Sample File (2)
Sample File (2)
Transform File (2)
RLS CC Lookup
Parameter8
Transform Sample File (8)
Sample File (8)
Transform File (8)
Parameter12
Transform Sample File (12)
Sample File (12)
Transform File (12)
Parameter18
Transform Sample File (18)
Sample File (18)
Transform File (18)
Parameter4
Transform Sample File (4)
Sample File (4)
Transform File (4)
Parameter7
Transform Sample File (7)
Sample File (7)
Transform File (7)
Parameter26
Transform Sample File (26)
Parameter27
Transform Sample File (27)
Errors in Date Hierarchy
Erro

In [ ]:
import re

def remove_indented_block(text, target_text):
    lines = text.splitlines(keepends=True)
    result = []
    skipping = False
    
    for line in lines:
        # Ignore empty lines or lines with only whitespace while deciding
        stripped_line = line.strip()
        
        # Calculate indentation level (number of leading spaces/tabs)
        indent_level = len(line) - len(line.lstrip())
        
        # If we encounter a line at the base level (no indent), stop skipping
        if indent_level == 0 and stripped_line:
            skipping = False
            
        # Check if this line triggers the block deletion
        if target_text in line:
            skipping = True
            continue  # Skip the trigger line itself
            
        # Append line only if we are not currently skipping an indented block
        if not skipping:
            result.append(line)
            
    return "".join(result)



In [ ]:
from pathlib import Path
import shutil

# User inputs
ROOT_FOLDER = Path(
    r"C:\Users\LINGRAM\Documents\PBI Test Files"
)

REPORT_NAME = "5 Year Look Back (1)"

# Derived paths
MODEL_PATH = ROOT_FOLDER / f"{REPORT_NAME}.SemanticModel"
REPORT_PATH = ROOT_FOLDER / f"{REPORT_NAME}.Report"

# Validate paths
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"SemanticModel folder not found:\n{MODEL_PATH}"
    )

# Core model files/folders
EXPRESSIONS_FILE = MODEL_PATH / "definition" / "expressions.tmdl"
TABLES_FOLDER = MODEL_PATH / "definition" / "tables"

# Future use
if not REPORT_PATH.exists():
    print(
        f"Warning: Report folder not found:\n{REPORT_PATH}"
    )

print(f"Using model: {MODEL_PATH.name}")


def remove_expression(text, target_expression):
    lines = text.splitlines(keepends=True)

    result = []
    skipping = False

    for line in lines:

        # Start skipping when we find the target expression
        if line.startswith("expression ") and target_expression in line:
            skipping = True
            continue

        # Stop skipping when we hit the next expression
        if skipping and line.startswith("expression "):
            skipping = False

        if not skipping:
            result.append(line)

    return "".join(result)


def find_expression_references(expressions, tables_folder):
    references = {
        expr: {
            "file_count": 0,
            "files": set()
        }
        for expr in expressions
    }

    for tmdl_file in Path(tables_folder).rglob("*.tmdl"):
        with open(tmdl_file, "r", encoding="utf-8") as f:
            contents = f.read()

        for expr in expressions:
            if expr in contents:
                references[expr]["files"].add(tmdl_file.name)

    for expr in references:
        references[expr]["file_count"] = len(
            references[expr]["files"]
        )

    return references


# Read expressions.tmdl
with open(EXPRESSIONS_FILE, "r", encoding="utf-8") as f:
    text = f.read()


# Extract expression names
expressions = []

for line in text.splitlines():
    if line.startswith("expression "):
        expr_name = line.split("=", 1)[0]
        expr_name = expr_name[len("expression "):].strip().strip("'")
        expressions.append(expr_name)


# Find references
references = find_expression_references(
    expressions,
    TABLES_FOLDER
)


# Build list of unused expressions
unused_expressions = []

for expr, info in references.items():
    if info["file_count"] == 0:
        unused_expressions.append(expr)


# Summary
total_expressions = len(expressions)
removed_expressions = len(unused_expressions)
remaining_expressions = total_expressions - removed_expressions

print("\n===== EXPRESSION CLEANUP SUMMARY =====")
print(f"Total Expressions:   {total_expressions}")
print(f"Expressions Removed: {removed_expressions}")
print(f"Expressions Kept:    {remaining_expressions}")

print("\nRemoved Expressions:")
for expr in sorted(unused_expressions):
    print(f"  - {expr}")

if total_expressions > 0:
    print(
        f"\nRemoved {removed_expressions} of {total_expressions} expressions "
        f"({removed_expressions / total_expressions:.1%})"
    )


# Create backup
backup_file = EXPRESSIONS_FILE + ".bak"
shutil.copy(EXPRESSIONS_FILE, backup_file)

print(f"\nBackup created: {backup_file}")


# Remove unused expressions
for expr in unused_expressions:
    text = remove_expression(text, expr)


# Overwrite expressions.tmdl
with open(EXPRESSIONS_FILE, "w", encoding="utf-8") as f:
    f.write(text)

print("\nCleanup complete.")


===== EXPRESSION CLEANUP SUMMARY =====
Total Expressions:   158
Expressions Removed: 122
Expressions Kept:    36

Removed Expressions:
  - Errors in Cost Center Hierarchy
  - Errors in Date Hierarchy
  - Errors in Final YTD
  - Errors in Final YTD (2)
  - Errors in New Hires
  - Parameter1
  - Parameter10
  - Parameter11
  - Parameter12
  - Parameter13
  - Parameter14
  - Parameter15
  - Parameter16
  - Parameter17
  - Parameter18
  - Parameter19
  - Parameter2
  - Parameter20
  - Parameter21
  - Parameter22
  - Parameter23
  - Parameter24
  - Parameter25
  - Parameter26
  - Parameter27
  - Parameter28
  - Parameter29
  - Parameter3
  - Parameter30
  - Parameter31
  - Parameter32
  - Parameter33
  - Parameter34
  - Parameter35
  - Parameter36
  - Parameter37
  - Parameter38
  - Parameter4
  - Parameter5
  - Parameter6
  - Parameter7
  - Parameter8
  - Parameter9
  - RLS CC Lookup
  - Sample File (10)
  - Sample File (11)
  - Sample File (12)
  - Sample File (16)
  - Sample File (18)
 